# AI Data Analyst Agent —  Notebook

Full design doc, architecture, and rubric mapping: see `01_architecture.md` in the repo.

**Phase 1** (architecture) is documented separately.
**Phase 2** starts below: generate synthetic e-commerce datasets.

In [1]:
!pip install -q langchain langchain-groq langgraph langchain-community \
    pandas numpy chromadb sentence-transformers

In [2]:
from google.colab import userdata
import os

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

# LangSmith tracing (enabled properly in Phase 14)
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = userdata.get("LANGCHAIN_API_KEY")
# os.environ["LANGCHAIN_PROJECT"] = "ai-data-analyst-agent"

print("GROQ key loaded:", bool(os.environ.get("GROQ_API_KEY")))

GROQ key loaded: True


## Phase 2: Generate synthetic e-commerce datasets

Data includes deliberate, discoverable patterns (a sales dip, a low-ROI
campaign, low-stock products) so agents built in later phases have real
signal to find — not random noise.

In [3]:
import numpy as np
import pandas as pd
from datetime import date, timedelta
import os

rng = np.random.default_rng(42)

In [4]:
categories = ["Electronics", "Home", "Apparel", "Beauty", "Sports"]
n_products = 30

product_rows = []
pid = 1
for cat in categories:
    for i in range(n_products // len(categories)):
        cost = round(rng.uniform(5, 150), 2)
        margin_mult = rng.uniform(1.4, 2.6)
        price = round(cost * margin_mult, 2)
        product_rows.append({
            "product_id": f"P{pid:03d}",
            "name": f"{cat[:4].upper()}-{i+1:02d}",
            "category": cat,
            "cost": cost,
            "price": price,
        })
        pid += 1

products = pd.DataFrame(product_rows)
star_electronics = products[products.category == "Electronics"]["product_id"].iloc[:2].tolist()
products.head()

,product_id,name,category,cost,price
0,P001,ELEC-01,Electronics,117.22,225.84
1,P002,ELEC-02,Electronics,129.50,289.67
2,P003,ELEC-03,Electronics,18.66,47.97
3,P004,ELEC-04,Electronics,115.37,270.34
4,P005,ELEC-05,Electronics,23.58,45.76


In [5]:
n_customers = 300
segments = rng.choice(["New", "Regular", "VIP"], size=n_customers, p=[0.35, 0.50, 0.15])
signup_start = date(2025, 1, 1)

customers = pd.DataFrame({
    "customer_id": [f"C{i:04d}" for i in range(1, n_customers + 1)],
    "signup_date": [signup_start + timedelta(days=int(rng.integers(0, 420))) for _ in range(n_customers)],
    "segment": segments,
})
customers.head()

,customer_id,signup_date,segment
0,C0001,2025-02-28,Regular
1,C0002,2026-01-27,Regular
2,C0003,2025-12-21,Regular
3,C0004,2025-02-20,Regular
4,C0005,2025-11-27,Regular


In [6]:
start_date = date(2026, 1, 1)
end_date = date(2026, 6, 30)
n_days = (end_date - start_date).days + 1

order_rows = []
order_id = 1
regions = ["Riyadh", "Jeddah", "Dammam", "Makkah", "Other"]
region_p = [0.35, 0.25, 0.15, 0.10, 0.15]

cust_segment_map = dict(zip(customers.customer_id, customers.segment))
cust_ids = customers.customer_id.tolist()
cust_weights = np.array([
    3.5 if cust_segment_map[c] == "VIP" else (1.5 if cust_segment_map[c] == "Regular" else 1.0)
    for c in cust_ids
])
cust_weights = cust_weights / cust_weights.sum()

for day_offset in range(n_days):
    current_date = start_date + timedelta(days=day_offset)
    month = current_date.month
    base_daily_orders = int(rng.integers(18, 30))

    for _ in range(base_daily_orders):
        cat = rng.choice(categories)
        cat_products = products[products.category == cat]

        # Intentional anomaly: Electronics stockout dip in March
        if cat == "Electronics" and month == 3:
            available = cat_products[~cat_products.product_id.isin(star_electronics)]
            if rng.random() < 0.55:
                continue  # simulate lost sales
            product = available.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]
        else:
            product = cat_products.sample(1, random_state=int(rng.integers(0, 1_000_000))).iloc[0]

        customer_id = rng.choice(cust_ids, p=cust_weights)
        qty = int(rng.integers(1, 4))
        if cust_segment_map[customer_id] == "VIP":
            qty += int(rng.integers(1, 3))

        order_rows.append({
            "order_id": f"O{order_id:05d}",
            "customer_id": customer_id,
            "product_id": product["product_id"],
            "date": current_date,
            "quantity": qty,
            "price": product["price"],
            "region": rng.choice(regions, p=region_p),
        })
        order_id += 1

orders = pd.DataFrame(order_rows)
orders.head()

,order_id,customer_id,product_id,date,quantity,price,region
0,O00001,C0019,P001,2026-01-01,3,225.84,Riyadh
1,O00002,C0056,P016,2026-01-01,4,289.37,Dammam
2,O00003,C0249,P030,2026-01-01,3,38.85,Jeddah
3,O00004,C0180,P026,2026-01-01,1,47.89,Riyadh
4,O00005,C0287,P016,2026-01-01,3,289.37,Makkah


In [7]:
inventory_rows = []
low_stock_targets = set(rng.choice(products.product_id, size=5, replace=False)) | set(star_electronics)

for _, p in products.iterrows():
    stock = int(rng.integers(0, 8)) if p.product_id in low_stock_targets else int(rng.integers(20, 200))
    inventory_rows.append({"product_id": p.product_id, "stock_level": stock, "reorder_threshold": 15})

inventory = pd.DataFrame(inventory_rows)
inventory.head()

,product_id,stock_level,reorder_threshold
0,P001,6,15
1,P002,1,15
2,P003,122,15
3,P004,30,15
4,P005,52,15


In [8]:
marketing_rows = []
months = pd.date_range(start_date, end_date, freq="MS")
campaign_names = {"Electronics": "Campaign_A", "Home": "Campaign_B",
                   "Apparel": "Campaign_C", "Beauty": "Campaign_D", "Sports": "Campaign_E"}

for m in months:
    for cat in categories:
        camp = campaign_names[cat]
        spend = round(float(rng.uniform(2000, 9000)), 2)

        if camp == "Campaign_D":
            # Intentional anomaly: high spend, weak conversions -> bad ROI
            spend = round(float(rng.uniform(6000, 12000)), 2)
            conversions = int(rng.integers(20, 60))
        elif camp == "Campaign_B":
            # Intentional anomaly: spend strongly correlates with conversions
            conversions = max(int(spend / 15 + rng.normal(0, 15)), 10)
        else:
            conversions = int(rng.integers(50, 220))

        marketing_rows.append({"campaign_id": camp, "category": cat, "month": m.strftime("%Y-%m"),
                                "spend": spend, "conversions": conversions})

marketing = pd.DataFrame(marketing_rows)
marketing.head()

,campaign_id,category,month,spend,conversions
0,Campaign_A,Electronics,2026-01,7605.23,109
1,Campaign_B,Home,2026-01,5912.70,415
2,Campaign_C,Apparel,2026-01,2370.22,62
3,Campaign_D,Beauty,2026-01,10172.87,44
4,Campaign_E,Sports,2026-01,6905.09,203


In [9]:
os.makedirs("data", exist_ok=True)
products.to_csv("data/products.csv", index=False)
customers.to_csv("data/customers.csv", index=False)
orders.to_csv("data/orders.csv", index=False)
inventory.to_csv("data/inventory.csv", index=False)
marketing.to_csv("data/marketing.csv", index=False)

print(f"orders: {len(orders)} | customers: {len(customers)} | products: {len(products)}")
print("Star electronics (out of stock in March):", star_electronics)

orders: 4160 | customers: 300 | products: 30
Star electronics (out of stock in March): ['P001', 'P002']


In [10]:
o = orders.merge(products[["product_id", "category"]], on="product_id")
o["revenue"] = o["quantity"] * o["price"]
o["month"] = pd.to_datetime(o["date"]).dt.strftime("%Y-%m")

print("Electronics revenue by month (expect a March dip):")
print(o[o.category == "Electronics"].groupby("month")["revenue"].sum().round(0))

d = marketing[marketing.campaign_id == "Campaign_D"]
print("\nCampaign_D ROI (conversions per $1000 spend):", round((d.conversions.sum() / d.spend.sum()) * 1000, 2))

b = marketing[marketing.campaign_id == "Campaign_B"]
print("Campaign_B spend-conversions correlation:", round(b.spend.corr(b.conversions), 3))

print("\nLow stock products (below reorder threshold):")
print(inventory[inventory.stock_level < inventory.reorder_threshold])

Electronics revenue by month (expect a March dip):
month
2026-01    59716.0
2026-02    44615.0
2026-03    22851.0
2026-04    57130.0
2026-05    55969.0
2026-06    53462.0
Name: revenue, dtype: float64

Campaign_D ROI (conversions per $1000 spend): 4.5
Campaign_B spend-conversions correlation: 0.987

Low stock products (below reorder threshold):
   product_id  stock_level  reorder_threshold
0        P001            6                 15
1        P002            1                 15
5        P006            5                 15
6        P007            4                 15
19       P020            3                 15
24       P025            4                 15


## Phase 3: Load & Inspect Data with Pandas

Verify the datasets are clean and joinable before building any tools on top
of them: dtypes, nulls, and referential integrity between tables.

In [11]:
orders = pd.read_csv("data/orders.csv")
customers = pd.read_csv("data/customers.csv")
products = pd.read_csv("data/products.csv")
inventory = pd.read_csv("data/inventory.csv")
marketing = pd.read_csv("data/marketing.csv")

orders["date"] = pd.to_datetime(orders["date"])
customers["signup_date"] = pd.to_datetime(customers["signup_date"])

print(orders.dtypes)

order_id               object
customer_id            object
product_id             object
date           datetime64[ns]
quantity                int64
price                 float64
region                 object
dtype: object


In [12]:
for name, df in [("orders", orders), ("customers", customers),
                  ("products", products), ("inventory", inventory),
                  ("marketing", marketing)]:
    n_nulls = df.isnull().sum().sum()
    print(f"{name}: shape={df.shape}, nulls={n_nulls}")

orders: shape=(4160, 7), nulls=0
customers: shape=(300, 3), nulls=0
products: shape=(30, 5), nulls=0
inventory: shape=(30, 3), nulls=0
marketing: shape=(30, 5), nulls=0


In [13]:
print("orders.product_id all valid:", orders["product_id"].isin(products["product_id"]).all())
print("orders.customer_id all valid:", orders["customer_id"].isin(customers["customer_id"]).all())
print("inventory covers exactly the product catalog:", set(inventory.product_id) == set(products.product_id))

orders.product_id all valid: True
orders.customer_id all valid: True
inventory covers exactly the product catalog: True


In [14]:
orders.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
order_id,4160,4160,O04144,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
customer_id,4160,300,C0219,42,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_id,4160,30,P030,159,NaN,NaN,NaN,NaN,NaN,NaN,NaN
date,4160,NaN,NaN,NaN,2026-04-01 05:42:41.538461440,2026-01-01 00:00:00,2026-02-13 00:00:00,2026-04-02 00:00:00,2026-05-18 00:00:00,2026-06-30 00:00:00,NaN
quantity,4160.0,NaN,NaN,NaN,2.417067,1.0,2.0,2.0,3.0,5.0,1.107732
price,4160.0,NaN,NaN,NaN,167.823262,38.85,96.36,162.98,250.63,360.26,90.171952
region,4160,5,Riyadh,1485,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [15]:
print(products.category.value_counts())
print()
print(customers.segment.value_counts())
print()
print(orders.region.value_counts())

category
Electronics    6
Home           6
Apparel        6
Beauty         6
Sports         6
Name: count, dtype: int64

segment
Regular    151
New        109
VIP         40
Name: count, dtype: int64

region
Riyadh    1485
Jeddah    1048
Other      614
Dammam     608
Makkah     405
Name: count, dtype: int64


## Phase 4: Data Analysis Tools

Real Pandas-backed functions, each returning a Pydantic model instead of a
raw dict or string. These become LangChain @tools in Phase 5 — testing them
standalone first means any bug is caught here, not hidden inside an agent.

In [16]:
from pydantic import BaseModel
from typing import Optional, List, Dict
import pandas as pd
import numpy as np

class RevenueResult(BaseModel):
    total_revenue: float
    order_count: int

class SalesGrowthResult(BaseModel):
    period1_revenue: float
    period2_revenue: float
    growth_pct: Optional[float]

class ProductPerformance(BaseModel):
    product_id: str
    name: str
    category: str
    revenue: float
    units: int

class CustomerMetricsResult(BaseModel):
    repeat_customer_rate_pct: float
    avg_order_value: float
    revenue_by_segment: Dict[str, float]

class InventoryRiskProduct(BaseModel):
    product_id: str
    name: str
    category: str
    stock_level: int
    reorder_threshold: int

class InventoryRiskResult(BaseModel):
    low_stock_count: int
    out_of_stock_count: int
    at_risk_products: List[InventoryRiskProduct]

class CampaignPerformance(BaseModel):
    campaign_id: str
    category: str
    total_spend: float
    total_conversions: int
    conversions_per_1000_spend: float

class CorrelationResult(BaseModel):
    campaign_id: str
    correlation: Optional[float]

In [17]:
orders = pd.read_csv("data/orders.csv", parse_dates=["date"])
customers = pd.read_csv("data/customers.csv", parse_dates=["signup_date"])
products = pd.read_csv("data/products.csv")
inventory = pd.read_csv("data/inventory.csv")
marketing = pd.read_csv("data/marketing.csv")

# orders.price and products.price would collide on merge (orders.price is
# the price actually paid; products.price is redundant here) — merge on
# product metadata only, never on products.price.
PRODUCT_META = products[["product_id", "name", "category"]]

def _with_category(df: pd.DataFrame) -> pd.DataFrame:
    d = df.merge(PRODUCT_META, on="product_id", how="left")
    d["revenue"] = d["quantity"] * d["price"]
    return d

In [18]:
def calculate_total_revenue(start_date: str = None, end_date: str = None,
                             category: str = None) -> RevenueResult:
    d = _with_category(orders)
    if start_date: d = d[d.date >= pd.Timestamp(start_date)]
    if end_date: d = d[d.date <= pd.Timestamp(end_date)]
    if category: d = d[d.category == category]
    return RevenueResult(total_revenue=round(float(d.revenue.sum()), 2), order_count=int(len(d)))


def calculate_sales_growth(period1_start: str, period1_end: str,
                            period2_start: str, period2_end: str,
                            category: str = None) -> SalesGrowthResult:
    r1 = calculate_total_revenue(period1_start, period1_end, category)
    r2 = calculate_total_revenue(period2_start, period2_end, category)
    growth = (r2.total_revenue - r1.total_revenue) / r1.total_revenue * 100 if r1.total_revenue else None
    return SalesGrowthResult(
        period1_revenue=r1.total_revenue,
        period2_revenue=r2.total_revenue,
        growth_pct=round(growth, 2) if growth is not None else None,
    )

In [19]:
def get_top_products(n: int = 5, by: str = "revenue") -> List[ProductPerformance]:
    d = _with_category(orders)
    agg = d.groupby(["product_id", "name", "category"]).agg(
        revenue=("revenue", "sum"), units=("quantity", "sum")).reset_index()
    agg = agg.sort_values("revenue" if by == "revenue" else "units", ascending=False)
    return [ProductPerformance(**row) for row in agg.head(n).to_dict("records")]


def get_bottom_products(n: int = 5, by: str = "revenue") -> List[ProductPerformance]:
    d = _with_category(orders)
    agg = d.groupby(["product_id", "name", "category"]).agg(
        revenue=("revenue", "sum"), units=("quantity", "sum")).reset_index()
    agg = agg.sort_values("revenue" if by == "revenue" else "units", ascending=True)
    return [ProductPerformance(**row) for row in agg.head(n).to_dict("records")]

In [20]:
def get_sales_by_category() -> Dict[str, float]:
    d = _with_category(orders)
    return d.groupby("category")["revenue"].sum().round(2).to_dict()


def get_sales_by_region() -> Dict[str, float]:
    d = orders.assign(revenue=orders.quantity * orders.price)
    return d.groupby("region")["revenue"].sum().round(2).to_dict()

In [21]:
def get_customer_metrics(segment: str = None) -> CustomerMetricsResult:
    d = orders.assign(revenue=orders.quantity * orders.price).merge(customers, on="customer_id")
    if segment: d = d[d.segment == segment]
    order_counts = d.groupby("customer_id").size()
    repeat_rate = round(float((order_counts > 1).mean() * 100), 2)
    aov = round(float(d.groupby("order_id")["revenue"].sum().mean()), 2)
    by_segment = d.groupby("segment")["revenue"].sum().round(2).to_dict()
    return CustomerMetricsResult(
        repeat_customer_rate_pct=repeat_rate,
        avg_order_value=aov,
        revenue_by_segment=by_segment,
    )

In [22]:
def get_inventory_risks() -> InventoryRiskResult:
    d = inventory.merge(PRODUCT_META, on="product_id")
    low = d[d.stock_level < d.reorder_threshold].sort_values("stock_level")
    out_of_stock = low[low.stock_level == 0]
    return InventoryRiskResult(
        low_stock_count=int(len(low)),
        out_of_stock_count=int(len(out_of_stock)),
        at_risk_products=[InventoryRiskProduct(**row) for row in
            low[["product_id", "name", "category", "stock_level", "reorder_threshold"]].to_dict("records")],
    )

In [23]:
def get_campaign_performance(campaign_id: str = None) -> List[CampaignPerformance]:
    d = marketing.copy()
    if campaign_id: d = d[d.campaign_id == campaign_id]
    agg = d.groupby(["campaign_id", "category"]).agg(
        total_spend=("spend", "sum"), total_conversions=("conversions", "sum")).reset_index()
    agg["conversions_per_1000_spend"] = round(agg.total_conversions / agg.total_spend * 1000, 2)
    return [CampaignPerformance(**row) for row in agg.to_dict("records")]


def calculate_correlation(campaign_id: str) -> CorrelationResult:
    d = marketing[marketing.campaign_id == campaign_id]
    corr = d.spend.corr(d.conversions)
    return CorrelationResult(campaign_id=campaign_id,
                              correlation=round(float(corr), 3) if pd.notna(corr) else None)

In [24]:
# Sanity-check every tool against the ground truth we built into the data in Phase 2

r = calculate_total_revenue(category="Electronics", start_date="2026-03-01", end_date="2026-03-31")
print("March Electronics revenue:", r)

g = calculate_sales_growth("2026-02-01", "2026-02-28", "2026-03-01", "2026-03-31", category="Electronics")
print("Feb->March Electronics growth:", g)
assert g.growth_pct < 0, "expected a real dip in March"

ir = get_inventory_risks()
print("\nInventory risks:", ir)
assert ir.low_stock_count == 6

cp = get_campaign_performance()
worst = min(cp, key=lambda x: x.conversions_per_1000_spend)
print("\nWorst-ROI campaign:", worst)
assert worst.campaign_id == "Campaign_D"

cc = calculate_correlation("Campaign_B")
print("\nCampaign_B correlation:", cc)
assert cc.correlation > 0.8

print("\nAll tool sanity checks passed.")

March Electronics revenue: total_revenue=22850.98 order_count=65
Feb->March Electronics growth: period1_revenue=44614.97 period2_revenue=22850.98 growth_pct=-48.78

Inventory risks: low_stock_count=6 out_of_stock_count=0 at_risk_products=[InventoryRiskProduct(product_id='P002', name='ELEC-02', category='Electronics', stock_level=1, reorder_threshold=15), InventoryRiskProduct(product_id='P020', name='BEAU-02', category='Beauty', stock_level=3, reorder_threshold=15), InventoryRiskProduct(product_id='P025', name='SPOR-01', category='Sports', stock_level=4, reorder_threshold=15), InventoryRiskProduct(product_id='P007', name='HOME-01', category='Home', stock_level=4, reorder_threshold=15), InventoryRiskProduct(product_id='P006', name='ELEC-06', category='Electronics', stock_level=5, reorder_threshold=15), InventoryRiskProduct(product_id='P001', name='ELEC-01', category='Electronics', stock_level=6, reorder_threshold=15)]

Worst-ROI campaign: campaign_id='Campaign_D' category='Beauty' total_

## Phase 5: Specialist Agents

Wrap the Phase 4 functions as LangChain tools, give each specialist agent
only the tools relevant to its domain, and let the LLM (via Groq) decide
which tool(s) to call for a given question — not hardcoded logic.

In [25]:
from typing import Optional
from langchain_core.tools import tool

@tool
def revenue_tool(start_date: Optional[str] = None, end_date: Optional[str] = None,
                  category: Optional[str] = None) -> dict:
    """Calculate total revenue and order count. Optionally filter by
    start_date/end_date (YYYY-MM-DD) and/or product category."""
    return calculate_total_revenue(start_date, end_date, category).model_dump()

@tool
def growth_tool(period1_start: str, period1_end: str, period2_start: str,
                 period2_end: str, category: Optional[str] = None) -> dict:
    """Compare revenue between two date periods and return the percentage
    growth (negative means a decline). Use this to answer 'why did sales
    change' style questions."""
    return calculate_sales_growth(period1_start, period1_end, period2_start, period2_end, category).model_dump()

@tool
def top_products_tool(n: int = 5, by: str = "revenue") -> list:
    """Return the top N best-selling products, ranked by 'revenue' or 'units'."""
    return [p.model_dump() for p in get_top_products(n, by)]

@tool
def bottom_products_tool(n: int = 5, by: str = "revenue") -> list:
    """Return the N worst-selling products, ranked by 'revenue' or 'units'."""
    return [p.model_dump() for p in get_bottom_products(n, by)]

@tool
def sales_by_category_tool() -> dict:
    """Return total revenue broken down by product category."""
    return get_sales_by_category()

@tool
def sales_by_region_tool() -> dict:
    """Return total revenue broken down by region."""
    return get_sales_by_region()

@tool
def customer_metrics_tool(segment: Optional[str] = None) -> dict:
    """Return repeat-customer rate, average order value, and revenue by
    customer segment. Optionally filter to one segment (New/Regular/VIP)."""
    return get_customer_metrics(segment).model_dump()

@tool
def inventory_risks_tool() -> dict:
    """Return products that are below their reorder threshold, including
    which ones are completely out of stock."""
    return get_inventory_risks().model_dump()

@tool
def campaign_performance_tool(campaign_id: Optional[str] = None) -> list:
    """Return spend, conversions, and conversions-per-$1000 for marketing
    campaigns. Optionally filter to one campaign_id."""
    return [c.model_dump() for c in get_campaign_performance(campaign_id)]

@tool
def correlation_tool(campaign_id: str) -> dict:
    """Return the correlation between marketing spend and conversions for
    a specific campaign_id, to assess whether spend is actually driving results."""
    return calculate_correlation(campaign_id).model_dump()

In [26]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)

print("LLM model:", llm.model_name)

LLM model: openai/gpt-oss-20b


In [27]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

def run_specialist_agent(tools: list, system_prompt: str, question: str, max_iterations: int = 5) -> str:
    """Generic tool-calling loop: the LLM decides which tools to call,
    we execute them, feed results back, repeat until it gives a final answer."""
    llm_with_tools = llm.bind_tools(tools)
    tool_map = {t.name: t for t in tools}
    messages = [SystemMessage(content=system_prompt), HumanMessage(content=question)]

    for _ in range(max_iterations):
        response = llm_with_tools.invoke(messages)
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for call in response.tool_calls:
            tool_fn = tool_map[call["name"]]
            result = tool_fn.invoke(call["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Agent did not reach a final answer within the iteration limit."

In [28]:
SALES_TOOLS = [revenue_tool, growth_tool, top_products_tool, bottom_products_tool,
               sales_by_category_tool, sales_by_region_tool]
CUSTOMER_TOOLS = [customer_metrics_tool]  # RAG retriever tool added in Phase 9
INVENTORY_TOOLS = [inventory_risks_tool]
MARKETING_TOOLS = [campaign_performance_tool, correlation_tool]

PREMISE_CHECK = """
Never assume the premise of a question is true. If your tools show the
data does NOT support the premise (e.g. the user asks why something
decreased but your data shows an increase, or asks about "this month"
but your tool only returns all-time totals with no date filter), say
so explicitly and clearly before offering any explanation. Do not
invent a causal story for something that did not happen."""

SALES_PROMPT = """You are the Sales Agent for an e-commerce data analyst system.
You analyze revenue, sales trends, top/bottom products, and category/regional
performance.

IMPORTANT: The order data available to you covers 2026-01-01 through 2026-06-30
only. Always use dates within this range unless the user explicitly names a
different year. Never guess or default to a different year.

Always call a tool to get real numbers before answering — never estimate or
guess a figure. If a tool returns zero revenue or zero orders, treat that as
a signal to double-check your date range before concluding there was no
activity. Cite the actual numbers returned by the tools.""" + PREMISE_CHECK

CUSTOMER_PROMPT = """You are the Customer Agent. You analyze customer segments,
repeat-purchase behavior, and customer value.

IMPORTANT: Your tool returns ALL-TIME aggregated totals — it has no month or
date filter. Never label its output as a specific month's data (e.g. "this
month"); describe it as overall/all-time figures instead.

Always call a tool to get real numbers before answering.""" + PREMISE_CHECK

INVENTORY_PROMPT = """You are the Inventory Agent. You identify stock risks:
low-stock and out-of-stock products.

IMPORTANT: Your tool returns a current stock SNAPSHOT, not a historical trend.
Do not claim it explains a change over time (e.g. "this month's" sales drop)
without saying clearly that this is a plausible contributing factor, not a
measured one.

Always call a tool to get real numbers before answering.""" + PREMISE_CHECK

MARKETING_PROMPT = """You are the Marketing Agent. You analyze campaign spend,
conversions, ROI, and the relationship between marketing spend and results.

IMPORTANT: Your tools return ALL-TIME aggregated totals across the full
2026-01 to 2026-06 range — they have no month filter. Never label output as
a specific month's data.

IMPORTANT: Whenever you identify a specific campaign (by campaign_id), you MUST
also state its associated product category by name (e.g., "Campaign D (Beauty
category)"). Other specialist agents key their data by category, not by
campaign_id, so omitting the category name breaks cross-agent and cross-turn
reasoning.

Always call a tool to get real numbers before answering.""" + PREMISE_CHECK

def sales_agent(question: str) -> str:
    return run_specialist_agent(SALES_TOOLS, SALES_PROMPT, question)

def customer_agent(question: str) -> str:
    return run_specialist_agent(CUSTOMER_TOOLS, CUSTOMER_PROMPT, question)

def inventory_agent(question: str) -> str:
    return run_specialist_agent(INVENTORY_TOOLS, INVENTORY_PROMPT, question)

def marketing_agent(question: str) -> str:
    return run_specialist_agent(MARKETING_TOOLS, MARKETING_PROMPT, question)

In [29]:
print(sales_agent("Why did Electronics sales decrease in March compared to February?"))

**Electronics sales in March 2026 were 48.78 % lower than in February 2026.**

| Period | Revenue (USD) |
|--------|---------------|
| Feb 2026 | **$44,614.97** |
| Mar 2026 | **$22,850.98** |

The growth calculation (using the `growth_tool`) shows a **‑48.78 %** change.  
The data itself does not provide a reason for the decline—only that the revenue dropped by roughly half. If you need to investigate potential causes (e.g., inventory issues, promotions, seasonality, or external events), you would need additional data such as order volume, product mix, or marketing activity for those months.


In [30]:
print(customer_agent("Which customer segment is most valuable to the business?"))

The **VIP** customer segment is the most valuable to the business.  
All‑time revenue by segment shows:

| Segment | Revenue |
|---------|---------|
| New | $332,223.24 |
| Regular | $651,422.20 |
| **VIP** | **$698,415.74** |

VIP customers generate the highest total revenue, making them the most valuable segment overall.


In [31]:
print(inventory_agent("Which products are at risk of running out of stock?"))

Here’s the current snapshot of products that are **at risk of running out of stock** (i.e., their stock level is below the reorder threshold):

| Product ID | Name      | Category     | Stock Level | Reorder Threshold |
|------------|-----------|--------------|-------------|-------------------|
| P002       | ELEC-02   | Electronics  | 1           | 15 |
| P020       | BEAU-02   | Beauty       | 3           | 15 |
| P025       | SPOR-01   | Sports       | 4           | 15 |
| P007       | HOME-01   | Home         | 4           | 15 |
| P006       | ELEC-06   | Electronics  | 5           | 15 |
| P001       | ELEC-01   | Electronics  | 6           | 15 |

**Summary**

- **Low‑stock products:** 6  
- **Out‑of‑stock products:** 0  

These six items are currently below their reorder thresholds and therefore are at risk of depleting if sales continue at the current pace. No products are currently out of stock.


In [32]:
print(marketing_agent("Which campaign has the worst return on marketing spend, and is spend actually correlated with conversions for our best campaign?"))

**Worst return on marketing spend**

- **Campaign D (Beauty category)** – conversions per $1,000 spent: **4.5**.  
  This is the lowest value among all campaigns, indicating the poorest return on spend.

**Correlation between spend and conversions for our best campaign**

- **Campaign B (Home category)** is the best performer, with the highest conversions per $1,000 spent (65.55).  
- The correlation tool reports a correlation of **0.987** for Campaign B, showing a very strong positive relationship between marketing spend and conversions.  

*All figures are aggregated totals for the period 2026‑01 to 2026‑06.*


## Phase 6: Manager / Router Agent

The Manager reads the user's question and decides which specialist agents
are relevant, using structured output (Pydantic), not keyword matching.
A question can route to more than one specialist at once.

In [33]:
from pydantic import BaseModel, Field

class RoutingDecision(BaseModel):
    needs_sales: bool = Field(description="True if the question involves revenue, sales trends, top/bottom products, or category/regional performance.")
    needs_customer: bool = Field(description="True if the question involves customer segments, repeat customers, or customer value.")
    needs_inventory: bool = Field(description="True if the question involves stock levels, low stock, or restocking.")
    needs_marketing: bool = Field(description="True if the question involves campaigns, marketing spend, conversions, or ROI.")
    reasoning: str = Field(description="One short sentence explaining the routing decision.")

MANAGER_PROMPT = """You are the Manager Agent for an e-commerce data analyst
system. You do not answer questions yourself. Your only job is to decide
which specialist agents should handle the user's question:

- Sales Agent: revenue, sales trends, top/bottom products, category/regional performance
- Customer Agent: customer segments, repeat customers, customer value
- Inventory Agent: stock levels, low stock, out-of-stock, restocking risk
- Marketing Agent: campaign spend, conversions, marketing ROI

A question can require more than one specialist. For broad questions like
'analyze my business' or 'what should I focus on this week', select ALL
relevant specialists rather than guessing which single one matters most."""

router_llm = llm.with_structured_output(RoutingDecision)

def route_question(question: str) -> RoutingDecision:
    return router_llm.invoke([
        {"role": "system", "content": MANAGER_PROMPT},
        {"role": "user", "content": question},
    ])

In [34]:
test_questions = [
    "Why did sales decrease this month?",
    "Which customer segments are most valuable to us?",
    "Which products are at risk of running out of stock?",
    "What should I focus on this week?",
    "Which campaign has the best ROI, and are our top customers buying more electronics?",
]

for q in test_questions:
    decision = route_question(q)
    print(f"Q: {q}")
    print(f"   -> sales={decision.needs_sales}, customer={decision.needs_customer}, "
          f"inventory={decision.needs_inventory}, marketing={decision.needs_marketing}")
    print(f"   reasoning: {decision.reasoning}\n")

Q: Why did sales decrease this month?
   -> sales=True, customer=True, inventory=True, marketing=True
   reasoning: The question asks for the cause of a sales decline, which could stem from changes in customer behavior, inventory shortages, marketing effectiveness, or overall sales trends. All four specialists should examine their respective areas to identify the root cause.

Q: Which customer segments are most valuable to us?
   -> sales=False, customer=True, inventory=False, marketing=False
   reasoning: The question asks about customer segments and their value, which falls under customer analysis.

Q: Which products are at risk of running out of stock?
   -> sales=False, customer=False, inventory=True, marketing=False
   reasoning: The user is asking about products at risk of running out of stock, which is an inventory issue.

Q: What should I focus on this week?
   -> sales=True, customer=True, inventory=True, marketing=True
   reasoning: The question is broad and seeks overall foc

In [36]:
def run_selected_specialists(question: str) -> dict:
    """Route the question, then call only the relevant specialists.
    Sequential for now — Phase 7 converts this into a parallel workflow."""
    decision = route_question(question)
    results = {}

    if decision.needs_sales:
        results["sales"] = sales_agent(question)
    if decision.needs_customer:
        results["customer"] = customer_agent(question)
    if decision.needs_inventory:
        results["inventory"] = inventory_agent(question)
    if decision.needs_marketing:
        results["marketing"] = marketing_agent(question)

    return {"routing": decision, "specialist_outputs": results}

# quick end-to-end test
out = run_selected_specialists("Why did sales decrease this month?")
print("Routed to:", list(out["specialist_outputs"].keys()))
for agent_name, answer in out["specialist_outputs"].items():
    print(f"\n=== {agent_name.upper()} ===\n{answer}")

Routed to: ['sales', 'inventory', 'marketing']

=== SALES ===
I’m sorry, but the data does not support the premise that sales decreased this month.

Using the growth comparison tool for the two one‑month periods:

| Period | Revenue (USD) |
|--------|---------------|
| May 2026 (2026‑05‑01 – 2026‑05‑31) | **281,925.52** |
| June 2026 (2026‑06‑01 – 2026‑06‑30) | **292,105.93** |

The tool reports a **+3.61 %** growth from May to June. In other words, sales actually increased by about $10,180.41 (≈ 3.6 %) during June, not decreased. If you have a different time frame or additional context you’d like to explore, let me know!

=== INVENTORY ===
I don’t have any sales figures for the current month, so I can’t confirm that sales actually fell.  
What I can tell you from the latest inventory snapshot is that **six products are below their reorder thresholds** (and none are completely out of stock). Those items are:

| Product ID | Name | Category | Stock Level | Reorder Threshold |
|---------

## Phase 7: Parallelization Workflow (LangGraph Functional API)

Convert the sequential specialist calls into real concurrent execution.
Each selected specialist runs as a @task; the @entrypoint launches all of
them at once and only blocks when it needs their results (.result()).

Workflow pattern: **Parallelization** — Sales, Customer, Inventory, and
Marketing analysis are independent given the same raw data, so none needs
to wait for another to start.

In [37]:
from langgraph.func import entrypoint, task
from langgraph.checkpoint.memory import InMemorySaver

# InMemorySaver is required for @entrypoint to run at all, and doubles as
# our short-term checkpointer (formalized properly in Phase 10).
checkpointer = InMemorySaver()

In [38]:
@task
def sales_task(question: str) -> str:
    return sales_agent(question)

@task
def customer_task(question: str) -> str:
    return customer_agent(question)

@task
def inventory_task(question: str) -> str:
    return inventory_agent(question)

@task
def marketing_task(question: str) -> str:
    return marketing_agent(question)

In [39]:
@entrypoint(checkpointer=checkpointer)
def parallel_analysis_workflow(question: str) -> dict:
    # Routing must finish first — we need the decision before we know
    # which tasks to launch. This part stays sequential; everything after
    # it runs concurrently.
    decision = route_question(question)

    # Launching a @task returns a future immediately — it does NOT block.
    # All of the calls below fire off in parallel; execution only blocks
    # when we call .result() on a future, which happens next.
    futures = {}
    if decision.needs_sales:
        futures["sales"] = sales_task(question)
    if decision.needs_customer:
        futures["customer"] = customer_task(question)
    if decision.needs_inventory:
        futures["inventory"] = inventory_task(question)
    if decision.needs_marketing:
        futures["marketing"] = marketing_task(question)

    # This is where we actually wait — but since all tasks were already
    # launched above, they've been running concurrently this whole time.
    results = {name: fut.result() for name, fut in futures.items()}

    return {
        "question": question,
        "routing": decision.model_dump(),
        "specialist_outputs": results,
    }

In [40]:
import time, uuid

config = {"configurable": {"thread_id": str(uuid.uuid4())}}

start = time.time()
result = parallel_analysis_workflow.invoke(
    "What should I focus on this week?", config
)
elapsed = time.time() - start

print(f"Routed to: {list(result['specialist_outputs'].keys())}")
print(f"Elapsed time: {elapsed:.1f}s\n")

for agent_name, answer in result["specialist_outputs"].items():
    print(f"=== {agent_name.upper()} ===\n{answer[:300]}...\n")

Routed to: ['sales', 'customer', 'inventory', 'marketing']
Elapsed time: 34.6s

=== SALES ===
Agent did not reach a final answer within the iteration limit....

=== CUSTOMER ===
**All‑time snapshot (no month‑specific data)**  
- **Repeat‑customer rate:** 99.33 %  
- **Average order value (AOV):** $404.34  
- **Revenue by segment:**  
  - New: $332,223.24  
  - Regular: $651,422.20  
  - VIP: $698,415.74  

Because the tool returns only all‑time aggregates, I can’t tell you ...

=== INVENTORY ===
Here’s a quick snapshot of the inventory risks that need your attention this week:

| Product ID | Name | Category | Stock Level | Reorder Threshold |
|------------|------|----------|-------------|-------------------|
| **P002** | ELEC‑02 | Electronics | **1** | 15 |
| **P020** | BEAU‑02 | Beauty | ...

=== MARKETING ===
**Overall picture (all‑time 2026‑01 to 2026‑06)**  

| Campaign | Category | Total Spend | Total Conversions | Conversions per $1,000 spend |
|----------|----------|-----------

## Phase 8: Aggregator Agent
Separates facts (tool-verified) from hypotheses, cross-references specialist
findings, produces final structured analysis.

In [41]:
from typing import List

class AggregatedAnalysis(BaseModel):
    facts: List[str] = Field(description="Numbers/statements directly confirmed by a specialist agent's tool calls.")
    findings: List[str] = Field(description="Important patterns noticed by cross-referencing multiple specialists.")
    possible_causes: List[str] = Field(description="Hypotheses, clearly labeled as unconfirmed unless two independent agents' tool data support the same cause.")
    recommendations: List[str] = Field(description="Concrete, actionable next steps.")
    confidence_and_limitations: str = Field(description="What data was missing/aggregated/unverifiable, and how that limits confidence.")

AGGREGATOR_PROMPT = """You are the Aggregator Agent. You receive outputs from
up to four specialist agents (Sales, Customer, Inventory, Marketing) that
answered the same business question independently.

Your job:
1. List FACTS: only things a specialist explicitly confirmed via a tool call with real numbers.
2. List FINDINGS: patterns you notice by cross-referencing two or more specialists' facts.
3. List POSSIBLE CAUSES: hypotheses. If only one specialist speculated without tool
   evidence, say so explicitly and mark it low-confidence. If two independent
   specialists' tool data point the same direction, say that convergence out loud.
4. List RECOMMENDATIONS: concrete actions.
5. State CONFIDENCE AND LIMITATIONS: note any specialist that said its data
   couldn't confirm/deny the question's premise, or any data that was
   all-time/aggregated rather than time-specific.

Never blend a hypothesis into the facts list."""

aggregator_llm = llm.with_structured_output(AggregatedAnalysis)

def aggregate_analysis(question: str, specialist_outputs: dict) -> AggregatedAnalysis:
    combined = "\n\n".join(f"=== {k.upper()} AGENT ===\n{v}" for k, v in specialist_outputs.items())
    return aggregator_llm.invoke([
        {"role": "system", "content": AGGREGATOR_PROMPT},
        {"role": "user", "content": f"Question: {question}\n\n{combined}"},
    ])

In [42]:
@task
def aggregator_task(question: str, specialist_outputs: dict) -> AggregatedAnalysis:
    return aggregate_analysis(question, specialist_outputs)

@entrypoint(checkpointer=checkpointer)
def full_analysis_workflow(question: str, *, previous: dict = None) -> dict:
    # Inject prior turn's Q&A into the current question so routing/specialists
    # actually have short-term context, not just an unused checkpointer.
    context_note = ""
    if previous is not None:
        context_note = (
            f"\n\n[Previous turn in this conversation]\n"
            f"Q: {previous['question']}\n"
            f"A: {json.dumps(previous['final_analysis'])}"
        )
    augmented_question = question + context_note

    decision = route_question(augmented_question)

    futures = {}
    if decision.needs_sales: futures["sales"] = sales_task(augmented_question)
    if decision.needs_customer: futures["customer"] = customer_task(augmented_question)
    if decision.needs_inventory: futures["inventory"] = inventory_task(augmented_question)
    if decision.needs_marketing: futures["marketing"] = marketing_task(augmented_question)

    specialist_outputs = {name: fut.result() for name, fut in futures.items()}
    final = aggregator_task(augmented_question, specialist_outputs).result()

    result = {
        "question": question,
        "routing": decision.model_dump(),
        "specialist_outputs": specialist_outputs,
        "final_analysis": final.model_dump(),
    }

    return entrypoint.final(
        value=result,
        save={"question": question, "final_analysis": final.model_dump()},
    )

In [45]:
config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = full_analysis_workflow.invoke("Why did sales decrease this month?", config)

import json
print(json.dumps(result["final_analysis"], indent=2))

{
  "facts": [
    "May\u202f2026 revenue: $281,925.52",
    "June\u202f2026 revenue: $292,105.93",
    "Growth from May to June: +3.61\u202f%",
    "Repeat\u2011customer rate: 99.33\u202f%",
    "Average order value: $404.34",
    "Revenue by segment: New $332,223.24; Regular $651,422.20; VIP $698,415.74",
    "Products below reorder thresholds: P002 (ELEC\u201102, Electronics, stock 1, threshold 15), P020 (BEAU\u201102, Beauty, stock 3, threshold 15), P025 (SPOR\u201101, Sports, stock 4, threshold 15), P007 (HOME\u201101, Home, stock 4, threshold 15), P006 (ELEC\u201106, Electronics, stock 5, threshold 15), P001 (ELEC\u201101, Electronics, stock 6, threshold 15)"
  ],
  "findings": [
    "Sales increased by 3.61\u202f% from May to June 2026, contradicting the premise of a monthly decline.",
    "Inventory shows six items below reorder thresholds, yet sales still rose, suggesting the low stock did not cause a dip in June.",
    "Customer metrics (high repeat\u2011customer rate and ave

## Phase 9: RAG Pipeline
Real load → split → embed → store → retrieve over policy documents,
wired into the Customer Agent's actual tool set (not standalone).

In [46]:
import os
os.makedirs("policies", exist_ok=True)

policies = {
"return_policy.txt": """Return Policy: Customers may return unused items within 30 days
of delivery for a full refund. Items must be in original packaging. Electronics
must be returned within 15 days due to rapid depreciation. Beauty products cannot
be returned once opened, for hygiene reasons. Refunds are processed within 5-7
business days to the original payment method.""",

"shipping_policy.txt": """Shipping Policy: Standard shipping takes 3-5 business days
within Riyadh, Jeddah, and Dammam, and 5-7 business days to other regions. Free
shipping applies to orders over $100. VIP segment customers receive free express
shipping (1-2 business days) regardless of order value.""",

"inventory_policy.txt": """Inventory Policy: Products are reordered automatically
when stock falls below the reorder threshold (default 15 units). High-velocity
categories (Electronics) use a lower threshold review cycle of 7 days; other
categories are reviewed every 14 days. Out-of-stock items are removed from
active promotion until restocked.""",

"discount_policy.txt": """Discount Policy: VIP customers receive an automatic 10%
loyalty discount on orders over $200. Seasonal campaigns may offer category-wide
discounts up to 20%, subject to marketing budget approval. Discounts cannot be
combined with free shipping promotions unless explicitly stated.""",

"customer_service_policy.txt": """Customer Service Policy: All support tickets
must receive a first response within 24 hours. VIP customers are routed to
priority support with a 4-hour response target. Escalations involving refund
disputes over $500 require manager approval before processing.""",
}

for filename, content in policies.items():
    with open(f"policies/{filename}", "w") as f:
        f.write(content)

print(f"Created {len(policies)} policy documents.")

Created 5 policy documents.


In [47]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

loader = DirectoryLoader("policies", glob="*.txt", loader_cls=TextLoader)
raw_docs = loader.load()

splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=40)
chunks = splitter.split_documents(raw_docs)

print(f"Loaded {len(raw_docs)} documents -> split into {len(chunks)} chunks")
print(chunks[0].page_content[:150])

/tmp/ipykernel_52335/2797436029.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader


Loaded 5 documents -> split into 7 chunks
Discount Policy: VIP customers receive an automatic 10%
loyalty discount on orders over $200. Seasonal campaigns may offer category-wide
discounts up 


In [48]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Clear any stale collection from previous re-runs before creating fresh
try:
    Chroma(collection_name="policies", embedding_function=embeddings).delete_collection()
except Exception:
    pass

vectorstore = Chroma.from_documents(chunks, embeddings, collection_name="policies")
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})
print("Vector store ready.")

/tmp/ipykernel_52335/4232033603.py:4: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_52335/4232033603.py:8: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  Chroma(collection_name="policies", embedding_function=embeddings).delete_collection()


Vector store ready.


In [49]:
test_query = "Can a customer return an electronics item after 20 days?"
retrieved = retriever.invoke(test_query)

for i, doc in enumerate(retrieved):
    print(f"--- chunk {i+1} ---")
    print(doc.page_content)
    print()

--- chunk 1 ---
Return Policy: Customers may return unused items within 30 days
of delivery for a full refund. Items must be in original packaging. Electronics
must be returned within 15 days due to rapid depreciation. Beauty products cannot

--- chunk 2 ---
be returned once opened, for hygiene reasons. Refunds are processed within 5-7
business days to the original payment method.

--- chunk 3 ---
Inventory Policy: Products are reordered automatically
when stock falls below the reorder threshold (default 15 units). High-velocity
categories (Electronics) use a lower threshold review cycle of 7 days; other
categories are reviewed every 14 days. Out-of-stock items are removed from



In [50]:
@tool
def policy_lookup_tool(question: str) -> str:
    """Search the company's policy documents (return, shipping, inventory,
    discount, customer service) for relevant text to answer a policy question."""
    docs = retriever.invoke(question)
    return "\n\n".join(d.page_content for d in docs)

# Rebuild Customer Agent's tool set to include RAG
CUSTOMER_TOOLS = [customer_metrics_tool, policy_lookup_tool]

def customer_agent(question: str) -> str:
    return run_specialist_agent(CUSTOMER_TOOLS, CUSTOMER_PROMPT, question)

@task
def customer_task(question: str) -> str:
    return customer_agent(question)

In [51]:
print(customer_agent(
    "A VIP customer wants to return an Electronics item they bought 20 days ago. "
    "Does our policy allow it, and how valuable is this customer segment to us?"
))

**Return eligibility**

Our return policy for electronics states that items must be returned within **15 days** of delivery for a full refund.  
Because the VIP customer purchased the item **20 days ago**, the return request falls outside the allowed window. Therefore, the return is **not permitted** under the current policy.

**VIP customer segment value (all‑time)**

| Metric | Value |
|--------|-------|
| Repeat‑customer rate | **100 %** |
| Average order value | **$576.73** |
| Total revenue from VIPs | **$698,415.74** |

These figures represent the overall, all‑time performance of the VIP segment.


## Phase 10: Short-Term Memory
Multi-turn conversation state within a single thread, using the LangGraph
checkpointer already wired into `full_analysis_workflow`. Demonstrates that
turn 2 can reference turn 1 without re-stating context.

In [52]:
# Reuse ONE thread_id across multiple invocations = short-term memory
stm_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

turn1 = full_analysis_workflow.invoke(
    "Which product category had the weakest marketing ROI?",
    stm_config
)
print("TURN 1 FINAL ANALYSIS:")
print(json.dumps(turn1["final_analysis"], indent=2))

TURN 1 FINAL ANALYSIS:
{
  "facts": [
    "Beauty category has the lowest conversions\u2011per\u2011$1,000 spend at 4.5, indicating the weakest marketing ROI among categories."
  ],
  "findings": [],
  "possible_causes": [
    "Low conversion rate for Beauty products, possibly due to ineffective targeting or creative; high cost per conversion; low product demand; seasonal factors; or competition driving up spend without proportional sales."
  ],
  "recommendations": [
    "1. Conduct a detailed audit of Beauty campaign performance (ad creatives, targeting, landing pages).",
    "2. Test alternative messaging and offers to improve conversion rates.",
    "3. Reallocate a portion of the Beauty budget to higher\u2011performing categories while maintaining a test budget for optimization.",
    "4. Analyze customer feedback and market trends to identify potential product or positioning gaps.",
    "5. Implement A/B testing on ad placements and bidding strategies to reduce cost per conversio

In [53]:
# Note: no need to restate "Campaign_D" or "Beauty" — the checkpointer
# holds prior state under stm_config's thread_id
turn2 = full_analysis_workflow.invoke(
    "What about that same category's inventory situation — any stockouts?",
    stm_config
)
print("TURN 2 FINAL ANALYSIS:")
print(json.dumps(turn2["final_analysis"], indent=2))

TURN 2 FINAL ANALYSIS:
{
  "facts": [
    "No out-of-stock items in Beauty category.",
    "One Beauty product (BEAU-02) has stock level 3 below reorder threshold 15.",
    "Total 6 products across all categories below reorder thresholds, only one in Beauty."
  ],
  "findings": [
    "Beauty category has no current stockouts but contains a product at risk of stockout.",
    "Overall inventory risk is low for Beauty relative to other categories."
  ],
  "possible_causes": [
    "High demand for BEAU-02 leading to rapid depletion.",
    "Reorder threshold set too high relative to typical sales volume.",
    "Supply chain delays preventing timely replenishment.",
    "Seasonal spike in Beauty product sales.",
    "Inventory management lag in updating stock levels."
  ],
  "recommendations": [],
  "confidence_and_limitations": "Inventory data confirms no stockouts but only one product below reorder threshold; no sales or demand data to confirm cause. Findings are limited to inventory snaps

In [54]:
# A DIFFERENT thread_id should NOT have access to turn 1/2 context
fresh_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
turn3 = full_analysis_workflow.invoke(
    "What about that same category?",  # ambiguous without prior context
    fresh_config
)
print("FRESH THREAD (should show confusion/generic routing, no memory of Beauty):")
print(json.dumps(turn3["routing"], indent=2))

FRESH THREAD (should show confusion/generic routing, no memory of Beauty):
{
  "needs_sales": true,
  "needs_customer": false,
  "needs_inventory": false,
  "needs_marketing": false,
  "reasoning": "The user is asking about a specific category, which typically involves sales performance metrics such as revenue, trends, and top/bottom products within that category."
}


## Phase 11: Long-Term Memory
Cross-session persistent facts using LangGraph's Store, namespaced per user,
independent of any single conversation thread. Demonstrates recall in a
brand-new thread after the fact was learned in an earlier, different thread.

In [55]:
from langgraph.store.memory import InMemoryStore

# In production this would be a persistent store (Postgres, Redis, etc.)
# InMemoryStore is fine for demo/capstone purposes — note this in your README
long_term_store = InMemoryStore()

USER_NAMESPACE = ("user_memories", "demo_user")

In [56]:
@tool
def save_memory_tool(fact: str) -> str:
    """Save an important fact learned about the user or business context
    that should be remembered across future, unrelated conversations."""
    key = str(uuid.uuid4())
    long_term_store.put(USER_NAMESPACE, key, {"fact": fact})
    return f"Saved to long-term memory: {fact}"

@tool
def recall_memory_tool(query: str) -> str:
    """Retrieve previously saved facts about the user or business context
    that might be relevant to the current question."""
    items = long_term_store.search(USER_NAMESPACE, query=query, limit=5)
    if not items:
        return "No relevant long-term memories found."
    return "\n".join(f"- {item.value['fact']}" for item in items)

In [57]:
MANAGER_TOOLS_WITH_MEMORY = [recall_memory_tool]  # consulted before routing

def route_question_with_memory(question: str) -> RoutingDecision:
    prior_context = recall_memory_tool.invoke(question)
    augmented_question = f"{question}\n\n[Relevant long-term context]\n{prior_context}"
    return manager_llm.invoke([
        {"role": "system", "content": MANAGER_PROMPT},
        {"role": "user", "content": augmented_question},
    ])

In [58]:
session_a_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
print(save_memory_tool.invoke(
    "The business's fiscal year starts in April, not January — always frame "
    "'this year' comparisons using an April-start calendar."
))

Saved to long-term memory: The business's fiscal year starts in April, not January — always frame 'this year' comparisons using an April-start calendar.


In [59]:
session_b_config = {"configurable": {"thread_id": str(uuid.uuid4())}}  # different thread!

recalled = recall_memory_tool.invoke("fiscal year")
print("RECALLED IN A NEW SESSION:")
print(recalled)

RECALLED IN A NEW SESSION:
- The business's fiscal year starts in April, not January — always frame 'this year' comparisons using an April-start calendar.


## Phase 12: Human-in-the-Loop
Before the system "places" a reorder for a low-stock product, the workflow
pauses with interrupt() and waits for explicit human approval via
Command(resume=...). Placing a purchase order is treated as an irreversible
business action that should never fire automatically from an LLM's output.

In [60]:
from langgraph.types import interrupt, Command

def place_reorder(product_id: str, quantity: int) -> str:
    """Simulates actually submitting a purchase order to a supplier.
    In production this would call a real procurement/supplier API — here
    it's the irreversible action the human gate protects."""
    return f"Purchase order submitted: {quantity} units of {product_id}. Order confirmed."

In [61]:
@entrypoint(checkpointer=checkpointer)
def analysis_with_approval_workflow(question: str, *, previous: dict = None) -> dict:
    # Step 1: run the normal parallel analysis + aggregation (Phases 7-8)
    analysis = full_analysis_workflow.invoke(question, {"configurable": {"thread_id": str(uuid.uuid4())}})
    recommendations = analysis["final_analysis"]["recommendations"]

    # Step 2: does any recommendation involve restocking/reordering?
    reorder_rec = next((r for r in recommendations if "reorder" in r.lower()
                         or "restock" in r.lower() or "replenish" in r.lower()), None)

    if reorder_rec is None:
        return {"analysis": analysis, "action_taken": None, "approval": None}

    # Step 3: PAUSE. Nothing below this line runs until a human resumes it.
    decision = interrupt({
        "reason": "A recommendation involves submitting a reorder/purchase order.",
        "recommendation": reorder_rec,
        "question_asked": question,
        "action_required": "Approve or reject placing this reorder.",
    })

    # Step 4: only reached after Command(resume=...) is called
    if decision.get("approved"):
        result = place_reorder(
            decision.get("product_id", "P002"),
            decision.get("quantity", 50),
        )
        return {"analysis": analysis, "action_taken": result, "approval": "approved"}
    else:
        return {"analysis": analysis, "action_taken": "Skipped — human rejected the reorder.",
                 "approval": "rejected"}

In [62]:
hitl_config = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = analysis_with_approval_workflow.invoke(
    "Which products are at risk of running out of stock and what should we do about it?",
    hitl_config
)

print("PAUSED STATE — this is what interrupt() returned to the caller:")
print(result)

PAUSED STATE — this is what interrupt() returned to the caller:
{'__interrupt__': [Interrupt(value={'reason': 'A recommendation involves submitting a reorder/purchase order.', 'recommendation': 'Increase safety stock by 20–30% and re‑evaluate reorder thresholds', 'question_asked': 'Which products are at risk of running out of stock and what should we do about it?', 'action_required': 'Approve or reject placing this reorder.'}, id='b136ddd6776ce3499aaa424ddb2c2fb5')]}


In [63]:
# Simulates a human reviewing the paused state above and approving it
approval_decision = {
    "approved": True,
    "product_id": "P002",   # the most critical low-stock item (stock=1, threshold=15)
    "quantity": 50,
}

final_result = analysis_with_approval_workflow.invoke(
    Command(resume=approval_decision),
    hitl_config  # SAME thread_id — resume continues the paused run
)

print("FINAL RESULT AFTER HUMAN APPROVAL:")
print(final_result)

FINAL RESULT AFTER HUMAN APPROVAL:
{'analysis': {'question': 'Which products are at risk of running out of stock and what should we do about it?', 'routing': {'needs_sales': False, 'needs_customer': False, 'needs_inventory': True, 'needs_marketing': False, 'reasoning': 'The user is asking about products at risk of running out of stock and how to address that, which directly concerns inventory levels and restocking decisions.'}, 'specialist_outputs': {'inventory': '**Current inventory snapshot (as of the latest data pull)**  \n\n| Product ID | Name | Category | Stock Level | Reorder Threshold | At‑Risk Status |\n|------------|------|----------|-------------|-------------------|----------------|\n| P002 | ELEC‑02 | Electronics | **1** | 15 | Low‑stock |\n| P020 | BEAU‑02 | Beauty | **3** | 15 | Low‑stock |\n| P025 | SPOR‑01 | Sports | **4** | 15 | Low‑stock |\n| P007 | HOME‑01 | Home | **4** | 15 | Low‑stock |\n| P006 | ELEC‑06 | Electronics | **5** | 15 | Low‑stock |\n| P001 | ELEC‑01 |

In [64]:
hitl_config_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result2 = analysis_with_approval_workflow.invoke(
    "Which products are at risk of running out of stock and what should we do about it?",
    hitl_config_2
)
print("PAUSED (run 2):", result2)

rejection_decision = {"approved": False}
final_result2 = analysis_with_approval_workflow.invoke(
    Command(resume=rejection_decision),
    hitl_config_2
)
print("\nFINAL RESULT AFTER HUMAN REJECTION:")
print(final_result2)

PAUSED (run 2): {'__interrupt__': [Interrupt(value={'reason': 'A recommendation involves submitting a reorder/purchase order.', 'recommendation': 'Place urgent replenishment orders for each at‑risk product, prioritizing those with the lowest stock levels.', 'question_asked': 'Which products are at risk of running out of stock and what should we do about it?', 'action_required': 'Approve or reject placing this reorder.'}, id='6595a7b1ac4b769396b487aca2657539')]}

FINAL RESULT AFTER HUMAN REJECTION:
{'analysis': {'question': 'Which products are at risk of running out of stock and what should we do about it?', 'routing': {'needs_sales': False, 'needs_customer': False, 'needs_inventory': True, 'needs_marketing': False, 'reasoning': 'The user is specifically asking about products at risk of running out of stock and how to address that issue, which falls under inventory management.'}, 'specialist_outputs': {'inventory': '**Current inventory snapshot (as of the latest data pull)**  \n\n| Prod

## Phase 13: Error Handling
Two distinct error-handling strategies, both required by the rubric:
1. Retry — a real `RetryPolicy` object on a task prone to transient failures
   (LLM API calls, which we've seen hit rate limits during testing).
2. Fallback/Validation — a tool that validates its input and returns a
   structured error instead of raising an unhandled exception when given
   bad data (e.g. an unknown campaign_id).

In [65]:
from langgraph.types import RetryPolicy

llm_retry_policy = RetryPolicy(
    max_attempts=4,
    initial_interval=2.0,
    backoff_factor=2.0,
    max_interval=30.0,
)

@task(retry_policy=llm_retry_policy)
def sales_task(question: str) -> str:
    return sales_agent(question)

@task(retry_policy=llm_retry_policy)
def customer_task(question: str) -> str:
    return customer_agent(question)

@task(retry_policy=llm_retry_policy)
def inventory_task(question: str) -> str:
    return inventory_agent(question)

@task(retry_policy=llm_retry_policy)
def marketing_task(question: str) -> str:
    return marketing_agent(question)

@task(retry_policy=llm_retry_policy)
def aggregator_task(question: str, specialist_outputs: dict) -> AggregatedAnalysis:
    return aggregate_analysis(question, specialist_outputs)

In [66]:
attempt_counter = {"count": 0}

@task(retry_policy=RetryPolicy(max_attempts=3, initial_interval=1.0, backoff_factor=2.0))
def flaky_task(question: str) -> str:
    attempt_counter["count"] += 1
    if attempt_counter["count"] < 3:
        print(f"  [attempt {attempt_counter['count']}] simulating a transient failure...")
        raise ConnectionError("Simulated transient API failure (e.g. rate limit / network blip)")
    print(f"  [attempt {attempt_counter['count']}] succeeded")
    return "Recovered after retries."

@entrypoint(checkpointer=checkpointer)
def retry_demo_workflow(question: str) -> str:
    return flaky_task(question).result()

demo_config = {"configurable": {"thread_id": str(uuid.uuid4())}}
result = retry_demo_workflow.invoke("test", demo_config)
print("\nFinal result:", result)
print("Total attempts made:", attempt_counter["count"])

  [attempt 1] simulating a transient failure...
  [attempt 2] simulating a transient failure...
  [attempt 3] succeeded

Final result: Recovered after retries.
Total attempts made: 3


In [67]:
from langchain_core.tools import tool

@tool
def correlation_tool(campaign_id: str) -> dict:
    """Return the correlation between marketing spend and conversions for
    a specific campaign_id, to assess whether spend is actually driving results."""
    # Validate BEFORE touching the data — an LLM can hallucinate a
    # campaign_id that doesn't exist, and .corr() on an empty slice would
    # silently return NaN instead of a clear error.
    valid_ids = marketing["campaign_id"].unique().tolist()
    if campaign_id not in valid_ids:
        return {
            "error": f"Unknown campaign_id '{campaign_id}'. Valid options are: {valid_ids}",
            "campaign_id": campaign_id,
            "correlation": None,
        }
    return calculate_correlation(campaign_id).model_dump()

In [68]:
print("Valid campaign_id:")
print(correlation_tool.invoke({"campaign_id": "Campaign_B"}))

print("\nHallucinated / invalid campaign_id (simulating an LLM mistake):")
print(correlation_tool.invoke({"campaign_id": "Campaign_Z"}))

Valid campaign_id:
{'campaign_id': 'Campaign_B', 'correlation': 0.987}

Hallucinated / invalid campaign_id (simulating an LLM mistake):
{'error': "Unknown campaign_id 'Campaign_Z'. Valid options are: ['Campaign_A', 'Campaign_B', 'Campaign_C', 'Campaign_D', 'Campaign_E']", 'campaign_id': 'Campaign_Z', 'correlation': None}


## Phase 14: LangSmith Observability
Enable real tracing on every LLM call and workflow run, then inspect an
actual trace to demonstrate what it revealed — not just that tracing was
turned on.

In [69]:
import os
from google.colab import userdata

api_key = userdata.get("LANGSMITH_API_KEY")
project_name = "ai-data-analyst-agent-capstone"

# LangSmith configuration
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_ENDPOINT"] = "https://api.smith.langchain.com"
os.environ["LANGSMITH_API_KEY"] = api_key
os.environ["LANGSMITH_PROJECT"] = project_name

# Legacy names used by some LangChain components
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = api_key
os.environ["LANGCHAIN_PROJECT"] = project_name

# IMPORTANT for Colab/Jupyter:
# Make LangSmith callbacks synchronous so traces are uploaded
# before the next cell queries LangSmith.
os.environ["LANGCHAIN_CALLBACKS_BACKGROUND"] = "false"

print("LANGCHAIN_TRACING_V2:", os.environ.get("LANGCHAIN_TRACING_V2"))
print("LANGSMITH_TRACING:", os.environ.get("LANGSMITH_TRACING"))
print("LANGCHAIN_CALLBACKS_BACKGROUND:", os.environ.get("LANGCHAIN_CALLBACKS_BACKGROUND"))
print("API key set:", bool(api_key))
print("Project:", project_name)

LANGCHAIN_TRACING_V2: true
LANGSMITH_TRACING: true
LANGCHAIN_CALLBACKS_BACKGROUND: false
API key set: True
Project: ai-data-analyst-agent-capstone


In [70]:
from langsmith import Client
from langchain_core.tracers import LangChainTracer
import uuid
import json

tracer = LangChainTracer(
    project_name="ai-data-analyst-agent-capstone"
)

trace_config = {
    "configurable": {
        "thread_id": str(uuid.uuid4())
    },
    "callbacks": [tracer],
}

traced_result = full_analysis_workflow.invoke(
    "Which campaign has the best ROI and are our top customers buying more electronics?",
    trace_config
)

print(json.dumps(traced_result["final_analysis"], indent=2))

from langchain_core.tracers.langchain import wait_for_all_tracers

wait_for_all_tracers()

print("Trace upload finished.")

{
  "facts": [
    "Campaign\u202fB has 65.55 conversions per $1,000 spent (Marketing Agent)."
  ],
  "findings": [
    "Only one campaign\u2019s ROI metric is available; no comparison to other campaigns. No data on top customers\u2019 electronics purchases."
  ],
  "possible_causes": [
    "None identified \u2013 lack of data prevents hypothesis formation."
  ],
  "recommendations": [
    "1. Retrieve campaign-level ROI data for all campaigns (e.g., conversions, spend, revenue) to compare ROI. 2. Obtain customer-level purchase data with product categories to assess electronics spend trends among top customers. 3. Once data is available, calculate ROI per campaign and analyze electronics purchase trends."
  ],
  "confidence_and_limitations": "The only confirmed data comes from the Marketing Agent, who reported a conversion rate for Campaign\u202fB. The Customer Agent could not provide any data, and no other specialist confirmed campaign ROI or customer electronics spending. Therefore, 

In [71]:
from langchain_core.tracers.langchain import wait_for_all_tracers

print("Waiting for LangSmith traces to finish uploading...")

wait_for_all_tracers()

print("Trace upload finished.")

Waiting for LangSmith traces to finish uploading...
Trace upload finished.


In [72]:
from langsmith import Client

client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"],
)

print("Checking LangSmith traces...")

runs = list(
    client.list_runs(
        project_name="ai-data-analyst-agent-capstone",
        limit=10
    )
)

print("Number of runs:", len(runs))

for run in runs:
    print("Name:", run.name)
    print("ID:", run.id)
    print("Type:", run.run_type)
    print("Status:", run.status)
    print("------")

Checking LangSmith traces...


/tmp/ipykernel_52335/2448107729.py:11: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  client.list_runs(


Number of runs: 10
Name: PydanticToolsParser
ID: 01a04057-c2e9-7312-88f6-c401a99461fe
Type: parser
Status: success
------
Name: ChatGroq
ID: 01a04057-c0ce-7cf2-8534-444380240fea
Type: llm
Status: success
------
Name: RunnableSequence
ID: 01a04057-c0cd-7b10-a75b-f8ceaae269a3
Type: chain
Status: success
------
Name: aggregator_task
ID: 01a04057-c0cc-7230-985a-4abb30f33825
Type: chain
Status: success
------
Name: ChatGroq
ID: 01a04057-bcd9-75d3-bf2d-b06839b7d946
Type: llm
Status: success
------
Name: campaign_performance_tool
ID: 01a04057-bcc9-7553-88a4-123031973cf5
Type: tool
Status: success
------
Name: ChatGroq
ID: 01a04057-b7d3-7743-af0b-7715663273dd
Type: llm
Status: success
------
Name: marketing_task
ID: 01a04057-b7d0-7781-b35f-2a0aa9e27ac5
Type: chain
Status: success
------
Name: ChatGroq
ID: 01a04057-b7ce-7d91-9abf-dbf0900b818e
Type: llm
Status: success
------
Name: customer_task
ID: 01a04057-b7cd-79e0-8fcc-0583f192f0c8
Type: chain
Status: success
------


In [73]:
from langsmith import traceable
import uuid

@traceable(
    name="AI Data Analyst Workflow Test",
    project_name="ai-data-analyst-agent-capstone"
)
def test_langsmith_trace():
    return {
        "test": "LangSmith tracing works",
        "thread_id": str(uuid.uuid4())
    }

test_result = test_langsmith_trace()

print(test_result)

{'test': 'LangSmith tracing works', 'thread_id': '7fa44165-3143-4782-af97-3ecf3693e0ff'}


In [74]:
from langsmith import Client

client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"],
)

runs = list(
    client.list_runs(
        project_name="ai-data-analyst-agent-capstone",
        limit=10
    )
)

print("Number of runs:", len(runs))

for run in runs:
    print("Name:", run.name)
    print("ID:", run.id)
    print("Type:", run.run_type)
    print("Status:", run.status)
    print("---")

/tmp/ipykernel_52335/2947994030.py:9: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  client.list_runs(


Number of runs: 10
Name: PydanticToolsParser
ID: 01a04057-c2e9-7312-88f6-c401a99461fe
Type: parser
Status: success
---
Name: ChatGroq
ID: 01a04057-c0ce-7cf2-8534-444380240fea
Type: llm
Status: success
---
Name: RunnableSequence
ID: 01a04057-c0cd-7b10-a75b-f8ceaae269a3
Type: chain
Status: success
---
Name: aggregator_task
ID: 01a04057-c0cc-7230-985a-4abb30f33825
Type: chain
Status: success
---
Name: ChatGroq
ID: 01a04057-bcd9-75d3-bf2d-b06839b7d946
Type: llm
Status: success
---
Name: campaign_performance_tool
ID: 01a04057-bcc9-7553-88a4-123031973cf5
Type: tool
Status: success
---
Name: ChatGroq
ID: 01a04057-b7d3-7743-af0b-7715663273dd
Type: llm
Status: success
---
Name: marketing_task
ID: 01a04057-b7d0-7781-b35f-2a0aa9e27ac5
Type: chain
Status: success
---
Name: ChatGroq
ID: 01a04057-b7ce-7d91-9abf-dbf0900b818e
Type: llm
Status: success
---
Name: customer_task
ID: 01a04057-b7cd-79e0-8fcc-0583f192f0c8
Type: chain
Status: success
---


In [75]:
# Test tracing the LangGraph workflow explicitly
from langsmith import traceable

@traceable(
    name="full_analysis_workflow",
    project_name="ai-data-analyst-agent-capstone"
)
def traced_workflow(question):
    return full_analysis_workflow.invoke(
        question,
        {"configurable": {"thread_id": str(uuid.uuid4())}}
    )

test_workflow_result = traced_workflow(
    "Which campaign has the best ROI and are our top customers buying more electronics?"
)

print(json.dumps(test_workflow_result["final_analysis"], indent=2))

{
  "facts": [
    "Campaign B (Home category) achieved 65.55 conversions per $1,000 spent, the highest among the five campaigns."
  ],
  "findings": [],
  "possible_causes": [],
  "recommendations": [
    "Request campaign\u2011level ROI data for the current period (e.g., last quarter) to confirm if Campaign B remains the best ROI.",
    "Obtain customer\u2011level purchase history broken down by product category to determine if top customers are increasing electronics purchases.",
    "If such data is unavailable, consider setting up a data collection process (e.g., tagging purchases by campaign and product category) for future analysis."
  ],
  "confidence_and_limitations": "The only numeric fact comes from the Marketing Agent and is an all\u2011time metric; no time\u2011specific or customer\u2011level data is available. The Customer Agent explicitly stated that it lacks the necessary data to answer the question about top customers buying electronics. Therefore, confidence in answer

In [76]:
from langsmith import Client

client = Client()
runs = list(client.list_runs(project_name="ai-data-analyst-agent-capstone", limit=1))
if runs:
    latest_run = runs[0]
    print(f"Latest trace URL: https://smith.langchain.com/o/-/projects/p/{latest_run.session_id}/r/{latest_run.id}")
else:
    print("No runs found yet — check LANGCHAIN_PROJECT name matches, and that tracing was enabled before invoke().")

/tmp/ipykernel_52335/1106273424.py:4: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  runs = list(client.list_runs(project_name="ai-data-analyst-agent-capstone", limit=1))


Latest trace URL: https://smith.langchain.com/o/-/projects/p/efead4b9-ac86-4c95-bb84-17632d450b5a/r/01a04057-c2e9-7312-88f6-c401a99461fe


In [77]:
from langsmith import Client
import uuid
from datetime import datetime, timezone

client = Client(
    api_url=os.environ["LANGSMITH_ENDPOINT"],
    api_key=os.environ["LANGSMITH_API_KEY"],
)

test_run_id = uuid.uuid4()
now = datetime.now(timezone.utc)

print("Creating a test run directly in LangSmith...")

client.create_run(
    id=test_run_id,
    name="DIRECT_LANGSMITH_TEST",
    run_type="chain",
    inputs={"message": "LangSmith direct write test"},
    project_name="ai-data-analyst-agent-capstone",
    start_time=now,
)

client.update_run(
    test_run_id,
    outputs={"result": "SUCCESS"},
    end_time=datetime.now(timezone.utc),
)

print("Direct run created:", test_run_id)

Creating a test run directly in LangSmith...
Direct run created: dd75d2ce-f961-4089-9644-6a3e3d437480


In [78]:
from langsmith import Client

runs = list(
    client.list_runs(
        project_name="ai-data-analyst-agent-capstone",
        limit=10
    )
)

print("Number of runs:", len(runs))

for run in runs:
    print("Name:", run.name)
    print("ID:", run.id)
    print("Type:", run.run_type)
    print("Status:", run.status)
    print("---")

/tmp/ipykernel_52335/2788992589.py:4: DeprecationWarning: list_runs() is deprecated and will be removed after Jan 31, 2027. Use client.runs.query() instead. See https://docs.langchain.com/langsmith/smithdb-sdk-migration#runs-query for the migration guide.
  client.list_runs(


Number of runs: 10
Name: PydanticToolsParser
ID: 01a04057-c2e9-7312-88f6-c401a99461fe
Type: parser
Status: success
---
Name: ChatGroq
ID: 01a04057-c0ce-7cf2-8534-444380240fea
Type: llm
Status: success
---
Name: RunnableSequence
ID: 01a04057-c0cd-7b10-a75b-f8ceaae269a3
Type: chain
Status: success
---
Name: aggregator_task
ID: 01a04057-c0cc-7230-985a-4abb30f33825
Type: chain
Status: success
---
Name: ChatGroq
ID: 01a04057-bcd9-75d3-bf2d-b06839b7d946
Type: llm
Status: success
---
Name: campaign_performance_tool
ID: 01a04057-bcc9-7553-88a4-123031973cf5
Type: tool
Status: success
---
Name: ChatGroq
ID: 01a04057-b7d3-7743-af0b-7715663273dd
Type: llm
Status: success
---
Name: marketing_task
ID: 01a04057-b7d0-7781-b35f-2a0aa9e27ac5
Type: chain
Status: success
---
Name: ChatGroq
ID: 01a04057-b7ce-7d91-9abf-dbf0900b818e
Type: llm
Status: success
---
Name: customer_task
ID: 01a04057-b7cd-79e0-8fcc-0583f192f0c8
Type: chain
Status: success
---
